# SupportPilot AI — Automated Customer Support Desk
### A sellable, production-shaped LangGraph application

---

## The problem this sells against

Small e-commerce stores receive 30–300 support emails a day. The owner answers them at
11pm, or pays $800–2500/month for an offshore VA, or lets them rot (and ~70% of
"where is my order?" / "what's your return policy?" emails need zero human judgment).
Slow replies kill repeat purchases; one mishandled refund kills a customer forever.

## What SupportPilot does

Every incoming message is automatically:

1. **Classified** — intent, sentiment, urgency (structured output, not regex)
2. **Routed** to a specialist agent:
   - 📦 **Order agent** — looks up real order data, answers shipping/status questions
   - 📚 **Knowledge agent** — answers policy/product questions from the store's FAQ
   - 💸 **Refund agent** — checks eligibility against policy, recommends a decision,
     then **freezes and waits for the owner's approval** before any money moves
   - 🚨 **Escalation** — angry customers get an instant empathetic acknowledgment
     and a ticket lands in the owner's queue
3. **Guard-railed** — every outgoing draft is checked against business rules
   (no invented discounts, no promises the store can't keep)
4. **Sent & logged** — full analytics trail of every ticket
5. **Remembered** — each customer has a persistent conversation thread; follow-up
   emails keep full context

## Architecture

```
START → intake → classify ──┬→ order_agent ────────┐
                            ├→ kb_agent ───────────┼→ guardrail → send_reply → END
                            ├→ general_agent ──────┘
                            ├→ refund_agent → human_approval ──approve──→ process_refund → send_reply
                            │                       └──deny──→ draft_denial → send_reply
                            └→ escalate → send_reply (ack + ticket)
```

## Why a customer pays for this

- Replies in seconds, 24/7, in the store's own voice and policies (all config, no code)
- Money-touching actions **never** run unattended — owner approves refunds from a queue
- Drop-in integrations: every external system below is an isolated `SWAP POINT`
  function (Shopify, Gmail, Zendesk, Postgres) — swap mocks for real APIs and ship

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3)

## 2. Business configuration — *the only thing you change per customer*

Selling this to a new store = editing this one dict. Tone, policies, escalation rules:
all prompt-injected from here. No code changes per client.

In [ ]:
BUSINESS = {
    "name": "TrailGear Co.",
    "products": "outdoor backpacks, totes, and travel accessories",
    "tone": "warm, brief, lightly outdoorsy. Sign off as 'The TrailGear Team'.",
    "refund_policy": {
        "window_days": 30,
        "conditions": "item must be delivered; refunds to original payment method in 5-7 business days",
    },
    "shipping_policy": "Free over $50. Standard 3-5 business days, express 1-2.",
    "forbidden": [
        "offering discounts or coupon codes",
        "promising exact delivery dates we don't have",
        "admitting legal fault",
    ],
    "escalation_email": "owner@trailgear.example",
}

## 3. Integrations layer — every external system in one place

Each function below is a **SWAP POINT**: mocked so the demo runs anywhere, with the
real implementation noted. This isolation is what makes the product deployable in a
day — the graph never knows whether data comes from a dict or from Shopify.

In [ ]:
import json as _json
from datetime import datetime

# ── SWAP POINT: order database ──────────────────────────────────────────
# Real life: Shopify Admin API / WooCommerce REST / your Postgres.
ORDERS = {
    "4521": {"item": "Blue Trail Backpack", "total": 89.99, "status": "delayed at warehouse",
             "eta": "2026-06-16", "delivered": False, "days_since_delivery": None},
    "4522": {"item": "Canvas Day Tote", "total": 34.50, "status": "delivered",
             "eta": None, "delivered": True, "days_since_delivery": 12},
    "4523": {"item": "Summit Duffel 60L", "total": 129.00, "status": "delivered",
             "eta": None, "delivered": True, "days_since_delivery": 45},
}

# ── SWAP POINT: FAQ / knowledge base ────────────────────────────────────
# Real life: replace keyword match with a vector store (see rag_basics.ipynb).
FAQ = [
    {"q": "shipping times cost", "a": BUSINESS["shipping_policy"]},
    {"q": "return refund policy window", "a": f"Returns accepted within {BUSINESS['refund_policy']['window_days']} days of delivery. {BUSINESS['refund_policy']['conditions']}."},
    {"q": "waterproof rain material", "a": "All TrailGear packs use 600D recycled ripstop with a DWR coating — rain-resistant, not submersible."},
    {"q": "warranty repair broken zipper strap", "a": "Lifetime warranty on stitching, zippers, and buckles. Email a photo and we repair or replace."},
    {"q": "international ship abroad customs", "a": "We ship to US and Canada only for now."},
]

# ── SWAP POINT: outbound email ──────────────────────────────────────────
# Real life: Gmail SMTP (this repo already has GMAIL_ADDRESS / GMAIL_APP_PASSWORD
# in .env — see updatedlangchain/jobs/job_application_sender.ipynb), or Zendesk API.
def deliver_email(to: str, body: str):
    print(f"  ── EMAIL to {to} " + "─" * 40)
    for line in body.splitlines():
        print(f"  │ {line}")
    print("  " + "─" * 56)

# ── SWAP POINT: refund execution ────────────────────────────────────────
# Real life: Stripe refund API / Shopify refund endpoint.
def execute_refund(order_id: str, amount: float) -> str:
    return f"REF-{order_id}-OK"

# ── SWAP POINT: ticket queue for the owner ──────────────────────────────
# Real life: Slack webhook, Trello, or a simple dashboard.
OWNER_QUEUE = []

# ── Analytics log (CSV-ready) ───────────────────────────────────────────
TICKET_LOG = []

def log_ticket(**row):
    row["timestamp"] = datetime.now().isoformat(timespec="seconds")
    TICKET_LOG.append(row)

## 4. The shared state

One persistent conversation per customer (`thread_id` = their email). Note
`intake` resets the *per-ticket* fields each time — the thread state survives
between emails, so stale drafts must be cleared explicitly.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class DeskState(TypedDict):
    messages: Annotated[list, add_messages]   # full customer history (appends)
    customer_email: str
    # per-ticket fields (reset by intake):
    intent: str
    sentiment: str
    urgency: str
    summary: str
    draft_reply: str
    refund_order_id: str
    refund_recommendation: str
    status: str

## 5. Classification — structured output, not string parsing

`with_structured_output` forces the LLM into a validated Pydantic schema.
No "hopefully it replied with one word" parsing — bad values are impossible.

In [ ]:
from pydantic import BaseModel, Field


class Triage(BaseModel):
    intent: Literal["order_status", "refund_request", "product_question", "complaint", "other"] = \
        Field(description="What the customer wants")
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="One-sentence summary of the request")


triage_llm = llm.with_structured_output(Triage)


def intake(state: DeskState):
    # reset per-ticket fields; conversation history stays
    return {"intent": "", "sentiment": "", "urgency": "", "summary": "",
            "draft_reply": "", "refund_order_id": "", "refund_recommendation": "", "status": ""}


def classify(state: DeskState):
    last = _text(state["messages"][-1].content)
    t = triage_llm.invoke(
        f"Triage this customer message for {BUSINESS['name']} ({BUSINESS['products']}):\n\n{last}"
    )
    print(f"  [triage] intent={t.intent} sentiment={t.sentiment} urgency={t.urgency}")
    return {"intent": t.intent, "sentiment": t.sentiment, "urgency": t.urgency, "summary": t.summary}

## 6. Tools — what the specialist agents can actually *do*

In [ ]:
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its ID. Returns item, total, shipping status, ETA."""
    order = ORDERS.get(order_id.strip().lstrip("#"))
    if not order:
        return f"No order found with ID {order_id}."
    return _json.dumps(order)


@tool
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order is eligible for a refund under store policy."""
    order = ORDERS.get(order_id.strip().lstrip("#"))
    if not order:
        return f"No order found with ID {order_id}."
    window = BUSINESS["refund_policy"]["window_days"]
    if not order["delivered"]:
        return "NOT YET ELIGIBLE: order not delivered. Customer may cancel instead."
    if order["days_since_delivery"] <= window:
        return f"ELIGIBLE: delivered {order['days_since_delivery']} days ago (within {window}-day window). Amount: ${order['total']}."
    return f"NOT ELIGIBLE: delivered {order['days_since_delivery']} days ago, outside the {window}-day window."


@tool
def search_faq(query: str) -> str:
    """Search the store's FAQ/knowledge base. Use keywords from the customer's question."""
    words = set(query.lower().split())
    scored = sorted(FAQ, key=lambda e: -len(words & set(e["q"].split())))
    hits = [e for e in scored if words & set(e["q"].split())][:2]
    if not hits:
        return "No FAQ entry found. Say you'll check with the team rather than guessing."
    return "\n".join(f"- {e['a']}" for e in hits)

## 7. Specialist agents

Each specialist is a focused ReAct agent (notebook 4's pattern via `create_agent`)
with only the tools and instructions for its job. The customer's full message history
goes in, so follow-ups ("what was my order number again?") just work.

In [ ]:
from langchain.agents import create_agent


def _text(content) -> str:
    """Gemini may return content as a string OR a list of content blocks — normalize."""
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)


VOICE = f"You write for {BUSINESS['name']}. Tone: {BUSINESS['tone']} Keep replies under 130 words."

order_specialist = create_agent(
    llm, [lookup_order],
    system_prompt=f"""{VOICE}
You handle order status questions. ALWAYS use lookup_order before answering — never guess.
If the customer gave no order ID, ask for it. Shipping policy: {BUSINESS['shipping_policy']}""",
)

kb_specialist = create_agent(
    llm, [search_faq],
    system_prompt=f"""{VOICE}
You answer product and policy questions. ALWAYS use search_faq first and answer ONLY
from what it returns. If the FAQ has nothing, say you'll check with the team.""",
)

refund_analyst = create_agent(
    llm, [lookup_order, check_refund_eligibility],
    system_prompt=f"""You are a refund analyst for {BUSINESS['name']}. Policy:
{BUSINESS['refund_policy']['window_days']}-day window. {BUSINESS['refund_policy']['conditions']}
Use your tools, then output EXACTLY this format:
RECOMMENDATION: APPROVE or DENY
ORDER_ID: <id or unknown>
DRAFT: <reply to the customer matching the recommendation. Tone: {BUSINESS['tone']}>""",
)


def _run_specialist(agent, state: DeskState) -> str:
    result = agent.invoke({"messages": state["messages"]})
    return _text(result["messages"][-1].content)


def order_agent(state: DeskState):
    return {"draft_reply": _run_specialist(order_specialist, state), "status": "auto"}


def kb_agent(state: DeskState):
    return {"draft_reply": _run_specialist(kb_specialist, state), "status": "auto"}


def general_agent(state: DeskState):
    reply = llm.invoke(
        [("system", VOICE + " Answer helpfully; if it's outside support scope, say so kindly.")]
        + state["messages"]
    )
    return {"draft_reply": _text(reply.content), "status": "auto"}


def refund_agent(state: DeskState):
    out = _run_specialist(refund_analyst, state)
    rec = "APPROVE" if "RECOMMENDATION: APPROVE" in out.upper() else "DENY"
    order_id = out.split("ORDER_ID:")[1].split("\n")[0].strip() if "ORDER_ID:" in out else "unknown"
    draft = out.split("DRAFT:")[1].strip() if "DRAFT:" in out else out
    return {"draft_reply": draft, "refund_recommendation": rec,
            "refund_order_id": order_id, "status": "needs_approval"}


def escalate(state: DeskState):
    # Angry customer: instant human-sounding acknowledgment + ticket for the owner
    ack = llm.invoke(
        [("system", VOICE + """ The customer is upset. Write a short, sincere acknowledgment:
apologize, say a human team member will reply within 4 business hours. Do NOT offer
refunds, discounts, or solutions yet.""")] + state["messages"]
    )
    ack = _text(ack.content)
    OWNER_QUEUE.append({
        "customer": state["customer_email"], "summary": state["summary"],
        "urgency": state["urgency"], "notify": BUSINESS["escalation_email"],
    })
    return {"draft_reply": ack, "status": "escalated"}

## 8. Guardrail + human approval + delivery

- **guardrail** — auto-replies are checked against the store's forbidden list and
  rewritten if they violate it. Cheap insurance against an LLM "being generous".
- **human_approval** — `interrupt()` freezes refunds until the owner decides
  (notebook 5's pattern). Money never moves unattended.

In [ ]:
def guardrail(state: DeskState):
    rules = "; ".join(BUSINESS["forbidden"])
    checked = llm.invoke(f"""You are a compliance checker. Rules — the reply must NOT involve: {rules}.
If this reply violates any rule, rewrite it minimally to comply. Otherwise return it UNCHANGED.
Return only the final reply text.

{state["draft_reply"]}""")
    return {"draft_reply": _text(checked.content)}


def human_approval(state: DeskState):
    decision = interrupt({
        "type": "refund_approval",
        "customer": state["customer_email"],
        "order_id": state["refund_order_id"],
        "ai_recommendation": state["refund_recommendation"],
        "draft_reply": state["draft_reply"],
        "instructions": "resume with {'action': 'approve'} or {'action': 'deny', 'reason': '...'}",
    })
    if decision["action"] == "approve":
        return {"status": "refund_approved"}
    return {"status": "refund_denied", "summary": decision.get("reason", "policy decision")}


def process_refund(state: DeskState):
    order = ORDERS.get(state["refund_order_id"], {})
    ref = execute_refund(state["refund_order_id"], order.get("total", 0))
    note = f"\n\n(Refund {ref} processed — funds arrive in 5-7 business days.)"
    return {"draft_reply": state["draft_reply"] + note}


def draft_denial(state: DeskState):
    reply = llm.invoke(f"""{VOICE}
The owner declined this refund request. Reason: {state["summary"]}.
Write a kind, firm reply explaining the decision and offering the lifetime
warranty repair option as an alternative where it makes sense.""")
    return {"draft_reply": _text(reply.content)}


def send_reply(state: DeskState):
    deliver_email(state["customer_email"], state["draft_reply"])
    log_ticket(customer=state["customer_email"], intent=state["intent"],
               sentiment=state["sentiment"], urgency=state["urgency"],
               status=state["status"] or "sent", summary=state["summary"])
    # record what we sent into the customer's permanent conversation history
    return {"messages": [("assistant", state["draft_reply"])]}

## 9. Wire the graph

In [ ]:
def route_by_triage(state: DeskState) -> str:
    if state["sentiment"] == "angry" or state["intent"] == "complaint":
        return "escalate"                       # anger overrides everything
    return {
        "order_status": "order_agent",
        "refund_request": "refund_agent",
        "product_question": "kb_agent",
        "other": "general_agent",
    }.get(state["intent"], "general_agent")


def route_after_approval(state: DeskState) -> str:
    return "process_refund" if state["status"] == "refund_approved" else "draft_denial"


builder = StateGraph(DeskState)
for name, fn in [("intake", intake), ("classify", classify), ("order_agent", order_agent),
                 ("kb_agent", kb_agent), ("general_agent", general_agent),
                 ("refund_agent", refund_agent), ("escalate", escalate),
                 ("guardrail", guardrail), ("human_approval", human_approval),
                 ("process_refund", process_refund), ("draft_denial", draft_denial),
                 ("send_reply", send_reply)]:
    builder.add_node(name, fn)

builder.add_edge(START, "intake")
builder.add_edge("intake", "classify")
builder.add_conditional_edges("classify", route_by_triage,
    ["order_agent", "kb_agent", "general_agent", "refund_agent", "escalate"])

for safe in ("order_agent", "kb_agent", "general_agent"):
    builder.add_edge(safe, "guardrail")
builder.add_edge("guardrail", "send_reply")

builder.add_edge("refund_agent", "human_approval")
builder.add_conditional_edges("human_approval", route_after_approval,
    ["process_refund", "draft_denial"])
builder.add_edge("process_refund", "send_reply")
builder.add_edge("draft_denial", "send_reply")

builder.add_edge("escalate", "send_reply")
builder.add_edge("send_reply", END)

# SWAP POINT: InMemorySaver → SqliteSaver/PostgresSaver in production
desk = builder.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import Image, display

try:
    display(Image(desk.get_graph().draw_mermaid_png()))
except Exception:
    print(desk.get_graph().draw_mermaid())

## 10. The inbox API — one function your email webhook calls

In [ ]:
def handle_message(customer_email: str, text: str):
    """Entry point. Real life: called by a Gmail/Zendesk webhook per incoming email."""
    print(f"\nINCOMING from {customer_email}: {text[:80]}")
    config = {"configurable": {"thread_id": customer_email}}
    result = desk.invoke(
        {"messages": [("user", text)], "customer_email": customer_email}, config
    )
    if "__interrupt__" in result:
        p = result["__interrupt__"][0].value
        print(f"  ⏸  WAITING FOR OWNER — refund on order {p['order_id']} "
              f"(AI recommends {p['ai_recommendation']})")
    return result


def owner_decision(customer_email: str, action: str, reason: str = ""):
    """Owner's approval queue. Real life: a button in a small dashboard."""
    config = {"configurable": {"thread_id": customer_email}}
    return desk.invoke(Command(resume={"action": action, "reason": reason}), config)

## 11. Demo — a morning of real support traffic

In [ ]:
# ① "Where is my order?" — order agent looks it up, replies instantly
handle_message("priya@example.com",
    "Hi, I ordered a blue backpack last week, order #4521. It still hasn't shipped?!");

In [ ]:
# ② Same customer follows up — MEMORY: no order number repeated, bot still knows
handle_message("priya@example.com",
    "Thanks! And once it arrives, how long do I have if I want to return it?");

In [ ]:
# ③ Product question from someone else — FAQ agent, isolated thread
handle_message("marco@example.com",
    "Is the Summit Duffel waterproof enough for kayak trips?");

In [ ]:
# ④ Refund inside the 30-day window — AI recommends APPROVE, then FREEZES
handle_message("lena@example.com",
    "I'd like a refund on order #4522, the tote is smaller than I expected.");

In [ ]:
# The owner clicks "approve" in their queue — refund executes, customer notified
owner_decision("lena@example.com", "approve");

In [ ]:
# ⑤ Refund OUTSIDE the window — AI recommends DENY, owner confirms with a reason
handle_message("sam@example.com", "Refund please for order #4523. Just got around to trying it.")
owner_decision("sam@example.com", "deny", reason="45 days since delivery, outside 30-day policy window");

In [ ]:
# ⑥ Angry customer — instant acknowledgment + ticket in the owner's queue
handle_message("dave@example.com",
    "This is the THIRD time I'm writing. My strap snapped on day two. Absolutely unacceptable!!")

print("\nOWNER QUEUE:")
for t in OWNER_QUEUE:
    print(f"  🚨 {t['urgency'].upper():6} {t['customer']}: {t['summary']}")

## 12. Analytics — what the owner sees at the end of the day

This log is the upsell: response times, deflection rate (% handled with zero human
minutes), sentiment trends. Pipe it to a Google Sheet or a dashboard.

In [ ]:
auto = sum(1 for r in TICKET_LOG if r["status"] in ("auto", "sent"))
print(f"Tickets handled : {len(TICKET_LOG)}")
print(f"Fully automatic : {auto}  ({auto / max(len(TICKET_LOG),1):.0%} deflection)")
print(f"Owner decisions : {sum(1 for r in TICKET_LOG if 'refund' in r['status'])}")
print(f"Escalations     : {sum(1 for r in TICKET_LOG if r['status'] == 'escalated')}")
print()
for r in TICKET_LOG:
    print(f"  {r['timestamp']}  {r['intent']:17} {r['sentiment']:8} {r['status']:15} {r['customer']}")

## 13. Taking this to a paying customer

**Deploy checklist (the 5 swap points):**

| Mock in this notebook | Production replacement | Effort |
|---|---|---|
| `ORDERS` dict | Shopify/WooCommerce API in `lookup_order` | ~1 hr |
| `FAQ` keyword list | Vector store over their real docs (see `rag_basics.ipynb`) | ~2 hrs |
| `deliver_email` print | Gmail SMTP (creds already in this repo's `.env`) or Zendesk | ~1 hr |
| `execute_refund` | Stripe/Shopify refund API | ~1 hr |
| `InMemorySaver` | `SqliteSaver` (one line) or Postgres | ~30 min |

Wrap `handle_message` in a FastAPI endpoint + an inbound email webhook, and
`owner_decision` behind two buttons in a one-page dashboard. That's the whole product.

**Pricing shapes that work for this:**
- Setup fee ($500–1500: connect their store, load their FAQ, tune the voice)
  + monthly ($99–299 by ticket volume)
- Or per-resolved-ticket pricing — the analytics log above is your invoice evidence

**The demo script for a sales call is section 11** — run cells ①–⑥ live:
instant order answer → memory on the follow-up → the refund freeze
("money never moves without you") → the angry-customer ticket. Those four moments
are what close the deal.

**Honest limits to state up front:** it answers only from their data (guardrailed,
but review the first week's transcripts together); add their edge-case policies to
`BUSINESS["forbidden"]` as you find them.